# ДЗ 2: диагностика запоминанием 16 пар

**Вопрос:** может ли полная текущая модель выучить 16 заданных программ? После 2000 шагов на 512 парах генерации остаются слабыми даже на train. Здесь уменьшаем набор, сохраняя модель и оптимизацию.

В Colab выбери **T4** и выполни все ячейки по порядку в новой сессии. Если нет PyArrow, выполни отдельную ячейку `%pip install pyarrow`. Скачивается только первый разрешённый train-файл; test не нужен. Прежний checkpoint не загружается: веса инициализируются с нуля, seed 42.

Первоначальный бюджет — **до 1000 обновлений**, batch 8. Та же модель 2+2/128, байты, обучаемые позиции, Post-LN, dropout 0.1, FP32, AdamW `lr=0.001`, clipping 1.0. Берём первые 16 разных вопросов из прежнего train-пула 512; программы не редактируем.

Каждые 50 шагов — loss и точность токенов на всех 16 парах без dropout. На шагах 250/500/750/1000 — greedy-генерация всех 16 программ с прежним cap 512. Проверяем совпадение текста и **всех токенов, включая EOS**. При точном воспроизведении 16/16 обучение заканчивается раньше. Синтаксис проверяется через AST; программы не исполняются. Запоминание не является оценкой обобщения или функциональной правильности исходных программ.

Результаты и checkpoint обновляются при каждом логировании: **`hw02-overfit-16-results.json`**, **`hw02-overfit-16.pt`**. После завершения сохрани notebook с outputs и JSON; checkpoint также можно скачать. Время включает диагностику и сохранение, поэтому его не сравниваем со временем прежнего обучающего цикла.

Если 16/16 воспроизводятся, следующий качественный опыт — расширение train. Если нет, смотрим динамику loss, точность и первое расхождение генерации; это повод исследовать обучение, а не автоматически увеличивать модель. В этом ноутбуке архитектурных изменений и ускорений нет. На момент подготовки GPU-прогон ещё не выполнен.

Источники: [ДЗ 2](https://github.com/ashaba1in/hse-nlp/blob/7f837940c4af9a686fca7bff5f02345218b6fe80/homeworks/hw2_code_generation.md), [датасет](https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/blob/b2724689ff079e9c22efd548fe134bfb29036dbb/README.md), [Karpathy: overfit одного batch, §2](https://karpathy.github.io/2019/04/25/recipe/).


In [ ]:
# HW2_OVERFIT_16: memorization diagnostic, full unchanged model.
import ast
import hashlib
import json
import math
import random
import time
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
import pyarrow.parquet as pq

SEED, MAX_STEPS, BATCH_SIZE = 42, 1000, 8
D_MODEL, N_LAYERS, MAX_POSITIONS = 128, 2, 2048
PAD, BOS, EOS, VOCAB = 0, 1, 2, 259  # 256 UTF-8 bytes + 3 special tokens
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Enable GPU in Colab before running this notebook'
device = torch.device('cuda')
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
print('PyTorch:', torch.__version__)


GENERATION_EVERY, GENERATION_CAP = 250, 512
RESULTS_FILE = Path('hw02-overfit-16-results.json')
CHECKPOINT_FILE = 'hw02-overfit-16.pt'


In [ ]:
# 1. Same allowed shard and train pool; first 16 distinct training questions.
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
if not (cache / 'train.parquet').exists():
    urlretrieve(f'{base_url}/data/train/part-00000.parquet', cache / 'train.parquet')
rows = next(pq.ParquetFile(cache / 'train.parquet').iter_batches(batch_size=4096)).to_pylist()
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
short_rows = [r for r in rows if len(r['question'].encode()) <= 512 and len(r['code'].encode()) < 512]
train_pool = [r for r in short_rows if not is_validation(r['question'])][:512]
pool_hash = hashlib.sha256(json.dumps(train_pool, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert pool_hash == '3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35', 'Data differs from previous baseline'
small_train, seen = [], set()
for row in train_pool:
    if row['question'] not in seen:
        small_train.append(row)
        seen.add(row['question'])
    if len(small_train) == 16:
        break
assert len(small_train) == len(seen) == 16
assert all(not is_validation(r['question']) for r in small_train)
for row in small_train:
    ast.parse(row['code'])
small_train_hash = hashlib.sha256(json.dumps(small_train, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
print('Rows:', len(small_train), 'distinct train questions from the same 512-row pool')
print('Target lengths incl. EOS:', [len(r['code'].encode()) + 1 for r in small_train])
print('Train SHA-256:', small_train_hash)


In [ ]:
# 2. Byte encoding: reversible, no tokenizer training or external vocabulary.
def encode(text):
    return [b + 3 for b in text.encode('utf-8')]
def decode(ids):
    return bytes(i - 3 for i in ids if 3 <= i < VOCAB).decode('utf-8', errors='replace')
def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS]) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c) for c in codes], batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS]) for c in codes], batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels



In [ ]:
# 3. Standard encoder-decoder: 2 + 2 layers, learned positions, ReLU, Post-LN.
class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print('Parameters:', sum(p.numel() for p in model.parameters()))



In [ ]:
# 4. Teacher forcing and greedy answer different questions.
@torch.inference_mode()
def evaluate_gold():
    model.eval()
    src, tgt, labels = training_batch(small_train)
    logits = model(src, tgt)
    predictions = logits.argmax(-1)
    keep = labels.ne(PAD)
    correct = predictions.eq(labels)
    return {
        'fixed_train_loss': criterion(logits.reshape(-1, VOCAB), labels.reshape(-1)).item(),
        'token_accuracy': correct[keep].float().mean().item(),
        'teacher_forced_sequence_exact_count': (correct | ~keep).all(-1).sum().item(),
        'eos_target_accuracy': correct[labels.eq(EOS)].float().mean().item(),
        'target_tokens_including_eos': keep.sum().item(),
    }

def inspect_generation(row, ids):
    active = [i for i in ids if i != PAD]
    expected = encode(row['code']) + [EOS]
    code = decode(active)
    mismatch = next((i for i, (a, b) in enumerate(zip(active, expected)) if a != b), None)
    if mismatch is None and len(active) != len(expected):
        mismatch = min(len(active), len(expected))
    try:
        tree = ast.parse(code)
        parses = True
        defines_solve = any(isinstance(node, ast.FunctionDef) and node.name == 'solve'
                            for node in tree.body)
    except (SyntaxError, ValueError):
        parses, defines_solve = False, False
    return {'question': row['question'], 'reference_code': row['code'], 'generated_code': code,
        'expected_tokens': len(expected), 'generated_tokens': len(active),
        'exact_code_match': code == row['code'], 'exact_tokens_with_eos': active == expected,
        'first_mismatch_token_position_0_based': mismatch,
        'eos_emitted': EOS in active,
        'early_eos': EOS in active and active.index(EOS) + 1 < GENERATION_CAP,
        'reached_cap': len(active) == GENERATION_CAP,
        'parses_as_python': parses, 'defines_solve': defines_solve}

@torch.inference_mode()
def evaluate_generation():
    model.eval()
    generated = model.generate(source_batch([r['question'] for r in small_train]),
                               max_new_tokens=GENERATION_CAP).cpu().tolist()
    examples = [inspect_generation(r, ids) for r, ids in zip(small_train, generated)]
    summary = {'examples': len(examples),
               'mean_generated_tokens': sum(e['generated_tokens'] for e in examples) / len(examples)}
    for metric in ['exact_code_match', 'exact_tokens_with_eos', 'eos_emitted',
                   'early_eos', 'reached_cap', 'parses_as_python', 'defines_solve']:
        summary[metric + '_count'] = sum(e[metric] for e in examples)
    return {'summary': summary, 'examples': examples}


In [ ]:
# 5. Same optimization; logging does not update weights or consume dropout RNG.
torch.cuda.synchronize()
started = time.perf_counter()
training_log, generation_log = [], []
config = {'seed': SEED, 'max_steps': MAX_STEPS, 'batch_size_train': BATCH_SIZE,
    'd_model': D_MODEL, 'encoder_layers': N_LAYERS, 'decoder_layers': N_LAYERS,
    'heads': 4, 'ffn_dim': 4 * D_MODEL, 'dropout': 0.1, 'norm_first': False,
    'lr': 0.001, 'weight_decay': optimizer.param_groups[0]['weight_decay'],
    'clip_norm': 1.0, 'scheduler': None, 'tokenizer': 'UTF-8 bytes',
    'max_positions': MAX_POSITIONS, 'generation_cap': GENERATION_CAP}
result = {'experiment': 'overfit-16', 'config': config,
    'gpu': torch.cuda.get_device_name(0), 'torch_version': torch.__version__,
    'dtype': str(next(model.parameters()).dtype),
    'parameters': sum(p.numel() for p in model.parameters()),
    'data_revision': revision, 'original_train_pool_sha256': pool_hash,
    'small_train_sha256': small_train_hash, 'training_examples': len(small_train),
    'selection': 'first 16 distinct questions from the unchanged 512-row train pool',
    'training_log': training_log, 'generation_log': generation_log,
    'memorization_passed': False, 'functional_quality_evaluated': False,
    'generalization_evaluated': False, 'run_finished': False}

def record(step, train_batch_loss=None, generate=False):
    entry = {'step': step, 'train_batch_loss': train_batch_loss, **evaluate_gold()}
    training_log.append(entry)
    print('LOSS', json.dumps(entry), flush=True)
    if generate:
        checked = {'step': step, **evaluate_generation()}
        generation_log.append(checked)
        result['memorization_passed'] = checked['summary']['exact_tokens_with_eos_count'] == len(small_train)
        print('GREEDY', json.dumps({'step': step, **checked['summary']}), flush=True)
    torch.save({'model': {k: v.cpu() for k, v in model.state_dict().items()},
                'optimizer': optimizer.state_dict(), 'steps': step, 'config': config,
                'small_train_sha256': small_train_hash}, CHECKPOINT_FILE)
    result['actual_steps'] = step
    result['wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - started
    RESULTS_FILE.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result['memorization_passed']

record(0)
for step in range(1, MAX_STEPS + 1):
    model.train()
    src, tgt, labels = training_batch(random.sample(small_train, BATCH_SIZE))
    logits = model(src, tgt)
    loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step == 1 or step % 50 == 0 or step == MAX_STEPS:
        generate = step % GENERATION_EVERY == 0 or step == MAX_STEPS
        if record(step, loss.item(), generate=generate):
            print('All 16 programs reproduced exactly with EOS; stopping.', flush=True)
            break
torch.cuda.synchronize()
result['run_finished'] = True
result['wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - started
RESULTS_FILE.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
# 6. Show failures first (or successes if all programs match).
last_generation = generation_log[-1]
examples = last_generation['examples']
failures = [e for e in examples if not e['exact_tokens_with_eos']]
for example in (failures or examples)[:3]:
    print('EXAMPLE', json.dumps(example, ensure_ascii=False), flush=True)
summary = {'actual_steps': result['actual_steps'], 'max_steps': MAX_STEPS,
    'memorization_passed': result['memorization_passed'],
    'last_teacher_forcing': training_log[-1], 'last_greedy': last_generation['summary'],
    'wall_seconds_including_diagnostics_and_checkpointing':
        result['wall_seconds_including_diagnostics_and_checkpointing']}
print('RESULT', json.dumps(summary), flush=True)
print('Saved:', str(RESULTS_FILE), CHECKPOINT_FILE)
print('Save notebook with outputs and download results JSON. Exact reproduction is not a generalization score.')
